# Telco Customer Churn: Who Should Get a Retention Offer?

INSY 674 Final Project

**Question.** Can we use a customer's account, service and billing information to find the customers who are likely to leave, and decide who should get a retention offer?

**How the notebook is organized (same order as the project checklist)**
1. Frame the problem and look at the big picture
2. Explore the data
3. Prepare the data
4. Shortlist promising models
5. Fine-tune the system (tuning, ensemble, cost-based threshold, error analysis, explanations)
6. Final evaluation on the test set
7. Conclusions and limitations

## 1. Frame the Problem and Look at the Big Picture

**Objective.** A telecom company loses revenue every time a customer cancels. The retention team can send offers (a discount, a free upgrade), but each offer costs money, so they cannot send one to everybody. Our model scores every customer by churn risk, and we then pick a cut-off that keeps the total cost as low as possible.

**Who uses it.** The retention team, once a month, on the full customer list. This is an offline batch job, so we do not need real-time predictions.

**Type of problem.** Supervised binary classification. The label `Churn` says whether the customer left in the last month (about 26.5% did).

**How we measure performance.**
- To compare models we use **PR-AUC** (average precision). Churners are the minority class, and PR-AUC shows how well a model ranks them without depending on a threshold. ROC-AUC, recall, precision and F1 are reported as well.
- To make the final decision we use **expected cost in dollars**. Missing a churner costs a lot more than sending an unnecessary offer, so the default 0.5 threshold is not the right choice.

**Cost assumptions** (examples to make the calculation concrete; the real numbers would come from the business):
- Each offer costs **$50**.
- A churner who gets an offer stays with probability **30%**.
- The revenue lost when a customer churns is **12 × MonthlyCharges**.

**Baselines.** A dummy model that always predicts the majority class, and a simple business rule: *month-to-month contract and tenure under 12 months = will churn*. A model is only worth using if it clearly beats the rule.

**Main assumption.** The patterns from this snapshot will still hold next month. We come back to this in the limitations.

In [ ]:
# Install the extra packages used in this notebook (only needed once)
%pip install -q kagglehub shap imbalanced-learn

In [ ]:
from pathlib import Path
import os
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_validate,
                                     cross_val_predict, RandomizedSearchCV, GridSearchCV,
                                     validation_curve)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, VotingClassifier
from sklearn.calibration import calibration_curve
from sklearn.metrics import (make_scorer, precision_score, recall_score, f1_score, accuracy_score,
                             roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay)
from scipy.stats import loguniform, randint

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## 2. Explore the Data

### 2.1 Load the data
We download the IBM Telco Customer Churn dataset from Kaggle with `kagglehub` (`pip install kagglehub` if needed). It has 7,043 customers and 21 columns.

In [ ]:
import kagglehub

DATA_DIR = Path(kagglehub.dataset_download("blastchar/telco-customer-churn"))
DATA_PATH = next(DATA_DIR.rglob("*.csv"))
data = pd.read_csv(DATA_PATH)
print("File:", DATA_PATH.name)
print("Shape:", data.shape)
data.head()

### 2.2 Clean the obvious issues
`TotalCharges` is stored as text. Eleven rows are blank, and we check below that all of them have `tenure = 0`. These are brand-new customers who have not been billed yet, so 0 is the correct value, not a missing one. `customerID` is just an identifier and is dropped. These steps are row-by-row fixes that do not learn anything from the data, so doing them before the split does not leak information.

In [ ]:
total_numeric = pd.to_numeric(data["TotalCharges"], errors="coerce")
blank = total_numeric.isna()
print("Blank TotalCharges:", blank.sum())
print("Tenure of those rows:", data.loc[blank, "tenure"].unique())
assert (data.loc[blank, "tenure"] == 0).all()

data["TotalCharges"] = total_numeric.fillna(0)
df = data.drop(columns="customerID")
df["Churn"] = (df["Churn"] == "Yes").astype(int)
assert df.isna().sum().sum() == 0
print("Clean shape:", df.shape)

### 2.3 Split first, then explore the training set only
We set aside 20% of customers as a test set before looking at any patterns, so that what we see in the test set cannot influence our choices. The split is stratified so both sets keep the same churn rate. The test set is used **once**, at the very end.

In [ ]:
X = df.drop(columns="Churn")
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print("Train:", X_train.shape, " churn rate:", round(y_train.mean(), 3))
print("Test: ", X_test.shape, " churn rate:", round(y_test.mean(), 3))

explore = X_train.assign(Churn=y_train)   # copy used for exploration only

### 2.4 Attribute profile
The table gives the type, number of distinct values and missing values for every column. Our notes on each group:

- **Target, `Churn`**: binary, 26.5% positive. The classes are imbalanced, so accuracy alone would be misleading (predicting "nobody churns" already gives about 73%).
- **Demographics, `gender`, `SeniorCitizen`, `Partner`, `Dependents`**: binary. Probably weak signals. `gender` should not drive decisions, so we will check that the model does not rely on it.
- **Tenure and billing, `tenure`, `MonthlyCharges`, `TotalCharges`**: numeric, no missing values after cleaning. `TotalCharges` is close to `tenure × MonthlyCharges`, so it mostly repeats information we already have. We test whether it is needed (ablation in section 4).
- **Contract and payment, `Contract`, `PaperlessBilling`, `PaymentMethod`**: categorical. `Contract` has a natural order (month-to-month < one year < two year), so it gets ordinal encoding. `PaymentMethod` has no order and gets one-hot encoding.
- **Services, `PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`**: categorical. "No internet service" and "No phone service" are not missing values. They just mean "No", and they repeat what `InternetService` and `PhoneService` already say, so we merge them into "No".
- **Noise and outliers**: none of the numeric columns has impossible values, and the charges are within reasonable ranges. The main data-quality issue was the blank `TotalCharges`, handled above.

In [ ]:
profile = pd.DataFrame({
    "dtype": explore.dtypes.astype(str),
    "unique_values": explore.nunique(),
    "missing": explore.isna().sum(),
})
display(profile)
display(explore.describe().T.round(2))

### 2.5 Visual exploration

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
explore["Churn"].map({0: "Stay", 1: "Churn"}).value_counts().plot.bar(ax=axes[0, 0], color=["#4C78A8", "#E45756"])
axes[0, 0].set(title="Target: customers who stayed vs churned", xlabel="", ylabel="Customers")
axes[0, 0].tick_params(axis="x", rotation=0)

for ax, col in zip([axes[0, 1], axes[1, 0], axes[1, 1]], ["tenure", "MonthlyCharges", "TotalCharges"]):
    sns.histplot(data=explore, x=col, hue="Churn", bins=30, stat="density",
                 common_norm=False, element="step", ax=ax)
    ax.set_title(f"{col} by churn")
plt.tight_layout()
plt.show()

New customers churn much more: the churners' tenure is concentrated in the first few months. Churners also tend to have higher monthly charges.

In [ ]:
eda_cols = ["Contract", "InternetService", "PaymentMethod", "OnlineSecurity", "TechSupport", "PaperlessBilling"]
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for col, ax in zip(eda_cols, axes.flat):
    rate = explore.groupby(col)["Churn"].mean().sort_values()
    rate.plot.barh(ax=ax, color="#E45756")
    ax.axvline(explore["Churn"].mean(), color="black", linestyle="--", linewidth=1)
    ax.set(title=f"Churn rate by {col}", xlabel="Churn rate", ylabel="")
plt.tight_layout()
plt.show()

for col in ["Contract", "InternetService", "PaymentMethod"]:
    summary = explore.groupby(col)["Churn"].agg(customers="size", churn_rate="mean")
    display(summary.sort_values("churn_rate", ascending=False).style.format({"churn_rate": "{:.1%}"}))

In [ ]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
corr = explore[num_cols + ["Churn"]].corr()
plt.figure(figsize=(6, 4.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Correlation between numeric attributes and churn")
plt.tight_layout()
plt.show()

### 2.6 What we learned from exploring
- **Contract type is the clearest signal.** Over 40% of month-to-month customers churn, compared with only a few percent of two-year customers.
- **The first year is the riskiest.** Churn is highest for low-tenure customers and drops steadily as tenure grows.
- **Fiber optic customers churn more than DSL customers**, even though fiber is the premium service. This may point to a price or quality problem, but the data cannot tell us which.
- **Electronic check stands out.** Customers paying by electronic check churn at more than double the rate of customers on automatic payments.
- **Protective add-ons go with lower churn.** Customers with `OnlineSecurity` or `TechSupport` churn much less.
- **`TotalCharges` is highly correlated with `tenure`**, so the two carry overlapping information.

These are associations, not causes. For example, month-to-month customers may simply be less committed to begin with.

**Extra data that would help:** dates for each customer snapshot (to build a proper "predict next month" setup), contract end dates, customer service calls and complaints, past offers and whether they worked, actual usage, and competitor prices in the customer's area.

**Transformations to try:** merge the redundant "No internet/phone service" categories, count the add-on services, flag automatic payment, group tenure into bands, and measure whether the customer's price has changed (next section).

## 3. Prepare the Data

All feature engineering is written as a scikit-learn transformer and placed **inside the pipeline**. During cross-validation every step is fit only on the training folds, so nothing leaks from the validation fold or the test set.

New features:
- `num_addons`: how many of the six add-on services the customer has.
- `price_change`: `TotalCharges − MonthlyCharges × tenure`. If the customer always paid today's price this is close to 0, so a large value suggests the price changed over time.
- `tenure_group`: tenure bands 0–12, 13–24, 25–48, 49–72 months.
- `auto_payment`: 1 if the customer pays by automatic bank transfer or credit card.

In [ ]:
ADDON_COLS = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
              "TechSupport", "StreamingTV", "StreamingMovies"]
TENURE_BINS = [-1, 12, 24, 48, 72]
TENURE_LABELS = ["0-12", "13-24", "25-48", "49-72"]


class TelcoFeatures(BaseEstimator, TransformerMixin):
    # Row-wise cleaning and feature engineering (learns nothing from the data).

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X = X.replace({"No internet service": "No", "No phone service": "No"})
        X["num_addons"] = (X[ADDON_COLS] == "Yes").sum(axis=1)
        X["price_change"] = X["TotalCharges"] - X["MonthlyCharges"] * X["tenure"]
        X["tenure_group"] = pd.cut(X["tenure"], bins=TENURE_BINS, labels=TENURE_LABELS).astype(str)
        X["auto_payment"] = X["PaymentMethod"].str.contains("automatic").astype(int)
        return X

Quick check on the training set that the new features carry signal:

In [ ]:
train_feat = TelcoFeatures().transform(X_train).assign(Churn=y_train)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
train_feat.groupby("num_addons")["Churn"].mean().plot.bar(ax=axes[0])
axes[0].set_title("Churn rate by number of add-ons")
train_feat.groupby("tenure_group")["Churn"].mean().reindex(TENURE_LABELS).plot.bar(ax=axes[1])
axes[1].set_title("Churn rate by tenure group")
train_feat.groupby("auto_payment")["Churn"].mean().plot.bar(ax=axes[2])
axes[2].set_title("Churn rate by automatic payment")
for ax in axes:
    ax.set_ylabel("Churn rate")
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

train_feat.groupby("Churn")["price_change"].describe().round(1)

Customers with automatic payment churn far less, and churn falls quickly with tenure. `price_change` is centered at 0 for both groups but has a wide spread, so it may help a little. The ablation in section 4 checks whether it is worth keeping.

**Encoding and scaling**
- Numeric columns are standardized (needed by logistic regression and KNN; harmless for trees).
- Binary Yes/No columns become a single 0/1 column.
- `Contract` is ordinal encoded (0, 1, 2) because its order matters.
- `InternetService`, `PaymentMethod` and `tenure_group` are one-hot encoded.

In [ ]:
NUM_COLS = ["tenure", "MonthlyCharges", "TotalCharges", "num_addons", "price_change"]
BIN_COLS = ["gender", "Partner", "Dependents", "PhoneService",
            "MultipleLines", "PaperlessBilling"] + ADDON_COLS
ORD_COLS = ["Contract"]
OHE_COLS = ["InternetService", "PaymentMethod", "tenure_group"]
PASS_COLS = ["SeniorCitizen", "auto_payment"]


def build_preprocessor(drop_cols=()):
    keep = lambda cols: [c for c in cols if c not in drop_cols]
    return ColumnTransformer([
        ("num", StandardScaler(), keep(NUM_COLS)),
        ("bin", OneHotEncoder(drop="if_binary", sparse_output=False), keep(BIN_COLS)),
        ("ord", OrdinalEncoder(categories=[["Month-to-month", "One year", "Two year"]]), keep(ORD_COLS)),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False), keep(OHE_COLS)),
        ("pass", "passthrough", keep(PASS_COLS)),
    ], verbose_feature_names_out=False).set_output(transform="pandas")


def make_model_pipeline(model, drop_cols=()):
    return Pipeline([
        ("features", TelcoFeatures()),
        ("prep", build_preprocessor(drop_cols)),
        ("model", model),
    ])


check = build_preprocessor().fit_transform(TelcoFeatures().transform(X_train))
print("Number of model inputs after preprocessing:", check.shape[1])
print(list(check.columns))

## 4. Shortlist Promising Models

### 4.1 Quick models from different families
We train fast, mostly default versions of models from different families: linear (logistic regression), probabilistic (naive Bayes), instance-based (KNN), a single tree, bagging (random forest) and boosting (histogram gradient boosting). We compare them with the dummy and rule baselines using **5-fold stratified cross-validation on the training set only** and report the mean and standard deviation of each metric. We also report the training-fold PR-AUC so we can spot overfitting.

In [ ]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
SCORING = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "recall": "recall",
    "precision": make_scorer(precision_score, zero_division=0),
    "f1": "f1",
    "accuracy": "accuracy",
}


class RuleBaseline(ClassifierMixin, BaseEstimator):
    # Business rule: month-to-month contract AND tenure < 12 months -> churn.

    def __init__(self, max_tenure=12):
        self.max_tenure = max_tenure

    def fit(self, X, y=None):
        self.classes_ = np.array([0, 1])
        return self

    def predict(self, X):
        rule = (X["Contract"] == "Month-to-month") & (X["tenure"] < self.max_tenure)
        return rule.astype(int).to_numpy()

    def predict_proba(self, X):
        p = self.predict(X)
        return np.column_stack([1 - p, p])


models = {
    "Dummy (majority class)": make_model_pipeline(DummyClassifier(strategy="prior")),
    "Rule baseline": RuleBaseline(),
    "Logistic Regression": make_model_pipeline(
        LogisticRegression(class_weight="balanced", max_iter=2000)),
    "Naive Bayes": make_model_pipeline(GaussianNB()),
    "KNN": make_model_pipeline(KNeighborsClassifier(n_neighbors=25)),
    "Decision Tree": make_model_pipeline(
        DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=RANDOM_STATE)),
    "Random Forest": make_model_pipeline(
        RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced",
                               random_state=RANDOM_STATE, n_jobs=-1)),
    "Gradient Boosting": make_model_pipeline(
        HistGradientBoostingClassifier(class_weight="balanced", random_state=RANDOM_STATE)),
}


def evaluate(models, X, y):
    rows = []
    for name, model in models.items():
        s = cross_validate(model, X, y, cv=CV, scoring=SCORING, return_train_score=True, n_jobs=-1)
        rows.append({
            "model": name,
            "PR-AUC": s["test_pr_auc"].mean(),
            "PR-AUC std": s["test_pr_auc"].std(),
            "ROC-AUC": s["test_roc_auc"].mean(),
            "ROC-AUC std": s["test_roc_auc"].std(),
            "Recall": s["test_recall"].mean(),
            "Precision": s["test_precision"].mean(),
            "F1": s["test_f1"].mean(),
            "Accuracy": s["test_accuracy"].mean(),
            "Train PR-AUC": s["train_pr_auc"].mean(),
        })
    return pd.DataFrame(rows).set_index("model").sort_values("PR-AUC", ascending=False).round(3)


results = evaluate(models, X_train, y_train)
results

In [ ]:
plot_df = results.drop(index=["Dummy (majority class)"]).sort_values("PR-AUC")
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(plot_df.index, plot_df["PR-AUC"], xerr=plot_df["PR-AUC std"], color="#4C78A8")
ax.axvline(y_train.mean(), color="black", linestyle="--", label=f"Random guess ({y_train.mean():.2f})")
ax.set(xlabel="Cross-validated PR-AUC (mean ± std)", title="Model comparison on the training set")
ax.legend()
plt.tight_layout()
plt.show()

**Reading the results.** All the real models beat the rule baseline by a wide margin. Logistic regression, random forest and gradient boosting are the top three, and their gaps are smaller than one standard deviation, so they are essentially tied for now. The tree ensembles score much higher on the training folds than on the validation folds, so they are overfitting and need regularization. Logistic regression scores about the same on both, so it is not overfitting. Naive Bayes and KNN are clearly weaker, and the single tree is limited by its depth.

In [ ]:
oof_proba = {name: cross_val_predict(m, X_train, y_train, cv=CV, method="predict_proba", n_jobs=-1)[:, 1]
             for name, m in models.items() if not name.startswith("Dummy")}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for name, p in oof_proba.items():
    if name == "Rule baseline":   # a rule gives one point, not a curve
        tp = ((p == 1) & (y_train == 1)).sum()
        rule_recall, rule_precision = tp / (y_train == 1).sum(), tp / (p == 1).sum()
        rule_fpr = ((p == 1) & (y_train == 0)).sum() / (y_train == 0).sum()
        axes[0].plot(rule_fpr, rule_recall, "ko", markersize=10, label="Rule baseline")
        axes[1].plot(rule_recall, rule_precision, "ko", markersize=10, label="Rule baseline")
        continue
    fpr, tpr, _ = roc_curve(y_train, p)
    prec, rec, _ = precision_recall_curve(y_train, p)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_train, p):.3f})")
    axes[1].plot(rec, prec, label=f"{name} (AP={average_precision_score(y_train, p):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", label="Random")
axes[0].set(xlabel="False positive rate", ylabel="Recall", title="ROC curve (out-of-fold, training set)")
axes[1].axhline(y_train.mean(), color="k", linestyle="--", label="Random")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision-recall curve (out-of-fold, training set)")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 4.2 Most significant variables
Before tuning, we check which inputs the top models rely on. For logistic regression we look at the coefficients (inputs are standardized, so their sizes are comparable). For the random forest we use the built-in importances.

In [ ]:
lr_quick = clone(models["Logistic Regression"]).fit(X_train, y_train)
rf_quick = clone(models["Random Forest"]).fit(X_train, y_train)
names = lr_quick.named_steps["prep"].get_feature_names_out()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
coefs = pd.Series(lr_quick.named_steps["model"].coef_[0], index=names).sort_values()
top = pd.concat([coefs.head(8), coefs.tail(8)])
top.plot.barh(ax=axes[0], color=np.where(top > 0, "#E45756", "#54A24B"))
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set(title="Logistic regression coefficients\n(red raises churn risk, green lowers it)")

imp = pd.Series(rf_quick.named_steps["model"].feature_importances_, index=names).sort_values().tail(15)
imp.plot.barh(ax=axes[1], color="#4C78A8")
axes[1].set(title="Random forest feature importance (top 15)")
plt.tight_layout()
plt.show()

Both models agree on the main drivers: contract type, tenure, fiber optic internet, charges and electronic check payment. `gender` is near the bottom for both, which is what we want.

### 4.3 Quick feature selection (ablation)
`TotalCharges` mostly repeats `tenure × MonthlyCharges`, and `auto_payment` repeats `PaymentMethod`. We test a "lean" version without them.

In [ ]:
LEAN_DROP = ["TotalCharges", "auto_payment"]
ablation = {
    "LogReg (all features)": models["Logistic Regression"],
    "LogReg (lean)": make_model_pipeline(LogisticRegression(class_weight="balanced", max_iter=2000), LEAN_DROP),
    "LogReg (no engineered features)": make_model_pipeline(
        LogisticRegression(class_weight="balanced", max_iter=2000),
        ["num_addons", "price_change", "tenure_group", "auto_payment"]),
    "Random Forest (all features)": models["Random Forest"],
    "Random Forest (lean)": make_model_pipeline(
        RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced",
                               random_state=RANDOM_STATE, n_jobs=-1), LEAN_DROP),
}
evaluate(ablation, X_train, y_train)[["PR-AUC", "PR-AUC std", "ROC-AUC"]]

Dropping the redundant columns, or even all the engineered features, changes PR-AUC by less than the fold-to-fold noise. The information in this dataset is already captured by a handful of strong variables, so feature engineering alone will not move the score much. We keep the full feature set because the engineered features are easy to explain to the business and they are used again in the error analysis.

### 4.4 Overfitting check: decision tree depth

In [ ]:
depths = range(1, 16)
train_scores, valid_scores = validation_curve(
    make_model_pipeline(DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE)),
    X_train, y_train, param_name="model__max_depth", param_range=depths,
    cv=CV, scoring="average_precision", n_jobs=-1)

plt.figure(figsize=(9, 5))
plt.plot(depths, train_scores.mean(axis=1), "b-o", label="Training folds")
plt.plot(depths, valid_scores.mean(axis=1), "g-o", label="Validation folds")
plt.fill_between(depths, valid_scores.mean(axis=1) - valid_scores.std(axis=1),
                 valid_scores.mean(axis=1) + valid_scores.std(axis=1), color="g", alpha=0.15)
plt.xlabel("max_depth")
plt.ylabel("PR-AUC")
plt.title("Validation curve: decision tree")
plt.legend()
plt.show()

Validation PR-AUC peaks at a shallow depth and then drops, while the training score keeps rising. This is classic overfitting. A depth-3 tree is easy to read and shows the same story as the exploration:

In [ ]:
tree_pipe = make_model_pipeline(DecisionTreeClassifier(max_depth=3, class_weight="balanced",
                                                       random_state=RANDOM_STATE)).fit(X_train, y_train)
plt.figure(figsize=(20, 8))
plot_tree(tree_pipe.named_steps["model"], feature_names=tree_pipe.named_steps["prep"].get_feature_names_out(),
          class_names=["Stay", "Churn"], filled=True, rounded=True, fontsize=10, impurity=False, proportion=True)
plt.show()

### 4.5 Handling class imbalance
We compare three options for logistic regression: doing nothing, `class_weight="balanced"`, and SMOTE oversampling. SMOTE has to sit **inside** an imbalanced-learn pipeline so that synthetic customers are only created from the training folds. Otherwise it would leak.

In [ ]:
try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE

    imb_pipe = ImbPipeline([
        ("features", TelcoFeatures()),
        ("prep", build_preprocessor()),
        ("sampler", "passthrough"),
        ("model", LogisticRegression(max_iter=2000)),
    ])
    imb_grid = [
        {"sampler": ["passthrough"], "model__class_weight": [None, "balanced"]},
        {"sampler": [SMOTE(random_state=RANDOM_STATE)], "model__class_weight": [None]},
    ]
    imb_search = GridSearchCV(imb_pipe, imb_grid, scoring=SCORING, refit="pr_auc", cv=CV, n_jobs=-1)
    imb_search.fit(X_train, y_train)

    imb = pd.DataFrame(imb_search.cv_results_)
    imb["strategy"] = ["SMOTE" if not isinstance(s, str) else
                       ("class_weight = balanced" if cw == "balanced" else "nothing")
                       for s, cw in zip(imb["param_sampler"], imb["param_model__class_weight"])]
    display(imb.set_index("strategy")[["mean_test_pr_auc", "mean_test_roc_auc", "mean_test_recall",
                                       "mean_test_precision", "mean_test_accuracy"]].round(3))
except ImportError:
    print("imbalanced-learn is not installed: pip install imbalanced-learn")

All three have almost the same PR-AUC and ROC-AUC, so they rank customers equally well. What changes is recall versus precision at the default 0.5 threshold. Since we will choose our own threshold from costs later, the imbalance method matters very little here. We keep `class_weight="balanced"` because it is simpler than SMOTE and does not create synthetic customers.

**Shortlist.** We keep the three best families by cross-validated PR-AUC for tuning.

In [ ]:
candidates = ["Logistic Regression", "Naive Bayes", "KNN", "Decision Tree", "Random Forest", "Gradient Boosting"]
shortlist = [m for m in results.index if m in candidates][:3]
print("Shortlisted for tuning:", shortlist)

## 5. Fine-Tune the System

### 5.1 Hyperparameter tuning
We use `RandomizedSearchCV` instead of a full grid search because the random forest and boosting models have many hyperparameters and their ranges are wide. Sampling 15–25 combinations from continuous distributions covers the space much more cheaply than a grid. Every search uses the same 5 folds and optimizes PR-AUC. The test set is still untouched.

In [ ]:
search_spaces = {
    "Logistic Regression": (
        make_model_pipeline(LogisticRegression(class_weight="balanced", max_iter=3000)),
        {"model__C": loguniform(1e-3, 1e2)}, 15),
    "Random Forest": (
        make_model_pipeline(RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)),
        {"model__n_estimators": randint(200, 600),
         "model__max_depth": [None, 4, 6, 8, 12],
         "model__min_samples_leaf": randint(1, 30),
         "model__max_features": ["sqrt", 0.3, 0.5]}, 20),
    "Gradient Boosting": (
        make_model_pipeline(HistGradientBoostingClassifier(class_weight="balanced", random_state=RANDOM_STATE)),
        {"model__learning_rate": loguniform(0.01, 0.3),
         "model__max_depth": [None, 3, 4, 6],
         "model__max_leaf_nodes": randint(8, 64),
         "model__min_samples_leaf": randint(10, 100),
         "model__l2_regularization": loguniform(1e-3, 10)}, 25),
    "KNN": (
        make_model_pipeline(KNeighborsClassifier()),
        {"model__n_neighbors": randint(10, 150), "model__weights": ["uniform", "distance"]}, 15),
    "Decision Tree": (
        make_model_pipeline(DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE)),
        {"model__max_depth": randint(2, 12), "model__min_samples_leaf": randint(5, 100)}, 20),
    "Naive Bayes": (
        make_model_pipeline(GaussianNB()),
        {"model__var_smoothing": loguniform(1e-10, 1e-1)}, 15),
}

tuned, tuned_rows = {}, []
for name in shortlist:
    pipe, params, n_iter = search_spaces[name]
    search = RandomizedSearchCV(pipe, params, n_iter=n_iter, scoring="average_precision",
                                cv=CV, random_state=RANDOM_STATE, n_jobs=-1)
    search.fit(X_train, y_train)
    tuned[name] = search
    i = search.best_index_
    tuned_rows.append({"model": name,
                       "before tuning": results.loc[name, "PR-AUC"],
                       "after tuning": search.cv_results_["mean_test_score"][i],
                       "std": search.cv_results_["std_test_score"][i],
                       "best parameters": {k.replace("model__", ""): (round(v, 4) if isinstance(v, float) else v)
                                           for k, v in search.best_params_.items()}})
tuned_table = pd.DataFrame(tuned_rows).set_index("model")
tuned_table.round(3)

Tuning mostly helps the tree ensembles. The search picked larger `min_samples_leaf` values, which regularize the trees and reduce the overfitting we saw in section 4. Logistic regression barely changes because it has only one hyperparameter and was already well regularized.

### 5.2 Ensemble
The three tuned models are quite different (linear, bagging, boosting), so averaging their predicted probabilities (soft voting) might reduce their individual errors. We evaluate the ensemble on the same 5 folds.

In [ ]:
ensemble = make_model_pipeline(VotingClassifier(
    estimators=[(name.replace(" ", "_"), clone(tuned[name].best_estimator_.named_steps["model"]))
                for name in shortlist],
    voting="soft"))

ens_scores = cross_validate(ensemble, X_train, y_train, cv=CV, scoring=SCORING, n_jobs=-1)

compare_rows = [{"model": name, "PR-AUC": tuned_table.loc[name, "after tuning"],
                 "PR-AUC std": tuned_table.loc[name, "std"]} for name in shortlist]
compare_rows.append({"model": "Soft-voting ensemble", "PR-AUC": ens_scores["test_pr_auc"].mean(),
                     "PR-AUC std": ens_scores["test_pr_auc"].std()})
final_compare = pd.DataFrame(compare_rows).set_index("model").sort_values("PR-AUC", ascending=False)
final_compare.round(4)

**Choosing the final model.** We take the model with the highest cross-validated PR-AUC. If another model is within 0.005, which is well below one standard deviation, we treat them as tied and prefer the simpler one in this order: logistic regression, then a single tuned model, then the ensemble. A simpler model is easier to explain to the retention team and cheaper to maintain.

In [ ]:
SIMPLICITY = {"Logistic Regression": 0, "Naive Bayes": 1, "KNN": 1, "Decision Tree": 1,
              "Random Forest": 2, "Gradient Boosting": 2, "Soft-voting ensemble": 3}
top_score = final_compare["PR-AUC"].max()
near_top = final_compare[final_compare["PR-AUC"] >= top_score - 0.005]
best_name = min(near_top.index, key=lambda m: SIMPLICITY[m])
best_model = ensemble.fit(X_train, y_train) if best_name == "Soft-voting ensemble" else tuned[best_name].best_estimator_

print("Models within 0.005 of the best PR-AUC:", list(near_top.index))
print("Selected model:", best_name)

### 5.3 Are the probabilities trustworthy?
The cost calculation multiplies predicted probabilities by dollar values, so we check calibration on out-of-fold predictions. With `class_weight="balanced"` the probabilities are pushed upward, so we expect the curve to sit below the diagonal. That is fine for ranking customers, but it means the threshold must be learned from costs (next step) and not read as "a 50% chance".

In [ ]:
oof_best = cross_val_predict(best_model, X_train, y_train, cv=CV, method="predict_proba", n_jobs=-1)[:, 1]
frac_pos, mean_pred = calibration_curve(y_train, oof_best, n_bins=10)

plt.figure(figsize=(6, 5))
plt.plot(mean_pred, frac_pos, "o-", label=best_name)
plt.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")
plt.xlabel("Predicted churn probability")
plt.ylabel("Actual churn rate")
plt.title("Calibration (out-of-fold, training set)")
plt.legend()
plt.show()

### 5.4 Choosing the threshold by cost
Total cost = (number of offers × $50) + (revenue lost from churners). For a churner who gets an offer, we assume 30% are kept, so 70% of their 12-month revenue is still lost. For a churner who gets no offer, all of it is lost. We try every threshold from 0.01 to 0.99 on the **out-of-fold training predictions** and keep the cheapest one.

In [ ]:
OFFER_COST = 50        # $ per retention offer
SUCCESS_RATE = 0.30    # share of would-be churners who stay after an offer
CLV_MONTHS = 12        # months of revenue lost when a customer churns

y_arr = y_train.to_numpy()
value_train = X_train["MonthlyCharges"].to_numpy() * CLV_MONTHS


def total_cost(offer, y, value, offer_cost=OFFER_COST, success_rate=SUCCESS_RATE):
    # Offers sent x offer cost + revenue still lost from churners.
    offer = np.asarray(offer, dtype=bool)
    lost = y * value * np.where(offer, 1 - success_rate, 1)
    return offer.sum() * offer_cost + lost.sum()


thresholds = np.round(np.arange(0.01, 1.00, 0.01), 2)
cost_no_offer = total_cost(np.zeros(len(y_arr)), y_arr, value_train)
curve = pd.DataFrame({
    "threshold": thresholds,
    "cost": [total_cost(oof_best >= t, y_arr, value_train) for t in thresholds],
    "pct_targeted": [(oof_best >= t).mean() for t in thresholds],
})
curve["savings"] = cost_no_offer - curve["cost"]
best_row = curve.loc[curve["cost"].idxmin()]
BEST_THRESHOLD = float(best_row["threshold"])

print(f"Cost if we do nothing (training set): ${cost_no_offer:,.0f}")
print(f"Best threshold: {BEST_THRESHOLD:.2f}  |  customers targeted: {best_row['pct_targeted']:.1%}"
      f"  |  savings vs doing nothing: ${best_row['savings']:,.0f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(curve["threshold"], curve["savings"], linewidth=2)
axes[0].axvline(BEST_THRESHOLD, color="green", linestyle="--", label=f"Best = {BEST_THRESHOLD:.2f}")
axes[0].axvline(0.5, color="gray", linestyle=":", label="Default 0.5")
axes[0].set(xlabel="Decision threshold", ylabel="Savings vs no offers ($)", title="Savings by threshold")
axes[0].legend()
axes[1].plot(curve["threshold"], curve["pct_targeted"] * 100, color="tab:orange", linewidth=2)
axes[1].axvline(BEST_THRESHOLD, color="green", linestyle="--")
axes[1].set(xlabel="Decision threshold", ylabel="% of customers who get an offer", title="Share of customers targeted")
plt.tight_layout()
plt.show()

In [ ]:
def summarize(name, offer, y=y_arr, value=value_train):
    offer = np.asarray(offer, dtype=bool)
    tp = (offer & (y == 1)).sum()
    cost = total_cost(offer, y, value)
    no_offer = total_cost(np.zeros(len(y)), y, value)
    return {"strategy": name, "offers sent": int(offer.sum()), "% targeted": offer.mean(),
            "recall": tp / (y == 1).sum(), "precision": tp / offer.sum() if offer.sum() else np.nan,
            "total cost ($)": cost, "savings vs no offers ($)": no_offer - cost,
            "savings per 1,000 customers ($)": (no_offer - cost) / len(y) * 1000}


FMT = {"% targeted": "{:.1%}", "recall": "{:.1%}", "precision": "{:.1%}", "total cost ($)": "${:,.0f}",
       "savings vs no offers ($)": "${:,.0f}", "savings per 1,000 customers ($)": "${:,.0f}"}

oof_rule = RuleBaseline().fit(X_train, y_train).predict(X_train)
expected_loss = oof_best * value_train
ev_thresholds = np.unique(np.quantile(expected_loss, np.linspace(0, 1, 201)))
best_ev_t = ev_thresholds[int(np.argmin([total_cost(expected_loss >= t, y_arr, value_train) for t in ev_thresholds]))]

strategies = pd.DataFrame([
    summarize("No offers", np.zeros(len(y_arr))),
    summarize("Offer to everyone", np.ones(len(y_arr))),
    summarize("Rule baseline", oof_rule),
    summarize("Model, threshold 0.50", oof_best >= 0.5),
    summarize(f"Model, threshold {BEST_THRESHOLD:.2f}", oof_best >= BEST_THRESHOLD),
    summarize("Model, rank by expected $ loss", expected_loss >= best_ev_t),
]).set_index("strategy")
display(strategies.style.format(FMT))

Using the cost-based threshold saves roughly twice as much as the rule baseline or offering to everyone. We also tried ranking customers by *expected dollar loss* (probability × customer value) instead of probability alone. Its savings are very close, so we keep the simpler probability threshold.

### 5.5 What if the business wants a minimum recall?
Our goal is to catch churners, so a natural question is: why not just maximize recall? Recall alone can always be pushed to 100% by sending an offer to everyone, and that is not the cheapest option. A fairer version is a constraint: *catch at least X% of churners, and among those thresholds pick the cheapest one.* The table shows what each recall target costs compared with the unconstrained cost-optimal threshold.

This is also why we used PR-AUC, not recall, to choose and tune models. Recall at the default 0.5 threshold mostly reflects the threshold and the class weights, not how well the model ranks customers. In section 4.5 the same logistic regression went from about 54% to about 80% recall just by adding `class_weight="balanced"`, while PR-AUC stayed the same. Recall belongs to the threshold decision, which we make here.

In [ ]:
best_cost = curve["cost"].min()
recall_by_t = np.array([recall_score(y_arr, oof_best >= t) for t in thresholds])
rows = [{"rule": "Cost-optimal (no recall target)", "threshold": BEST_THRESHOLD,
         "recall": recall_score(y_arr, oof_best >= BEST_THRESHOLD),
         "precision": precision_score(y_arr, oof_best >= BEST_THRESHOLD),
         "% targeted": (oof_best >= BEST_THRESHOLD).mean(), "extra cost per 1,000 customers ($)": 0.0}]
for target in [0.80, 0.85, 0.90, 0.95, 0.97]:
    ok = curve[recall_by_t >= target]
    if ok.empty:
        continue
    r = ok.loc[ok["cost"].idxmin()]
    offer = oof_best >= r["threshold"]
    rows.append({"rule": f"Recall >= {target:.0%}, cheapest", "threshold": r["threshold"],
                 "recall": recall_score(y_arr, offer), "precision": precision_score(y_arr, offer),
                 "% targeted": offer.mean(),
                 "extra cost per 1,000 customers ($)": (r["cost"] - best_cost) / len(y_arr) * 1000})
recall_targets = pd.DataFrame(rows).set_index("rule")
display(recall_targets.style.format({"threshold": "{:.2f}", "recall": "{:.1%}", "precision": "{:.1%}",
                                     "% targeted": "{:.1%}", "extra cost per 1,000 customers ($)": "${:,.0f}"}))

Low recall targets are free, because the cost-optimal threshold already meets them. Higher targets mean sending offers to a much larger share of customers, most of whom would not have left, and the extra cost grows quickly. If the retention team has a firm recall requirement, the notebook can apply it directly. Without one, we use the cost-optimal threshold.

### 5.6 How sensitive is this to our assumptions?
The $50 and 30% are guesses, so we recompute the best threshold under other values.

In [ ]:
rows = []
for offer_cost in [25, 50, 100]:
    for success_rate in [0.2, 0.3, 0.5]:
        costs = [total_cost(oof_best >= t, y_arr, value_train, offer_cost, success_rate) for t in thresholds]
        i = int(np.argmin(costs))
        base = total_cost(np.zeros(len(y_arr)), y_arr, value_train, offer_cost, success_rate)
        rows.append({"offer cost ($)": offer_cost, "success rate": success_rate,
                     "best threshold": thresholds[i], "savings per 1,000 customers ($)": (base - costs[i]) / len(y_arr) * 1000})
sensitivity = pd.DataFrame(rows)
print("Best threshold")
display(sensitivity.pivot(index="offer cost ($)", columns="success rate", values="best threshold"))
print("Savings per 1,000 customers ($)")
display(sensitivity.pivot(index="offer cost ($)", columns="success rate", values="savings per 1,000 customers ($)").round(0))

The best threshold moves a lot. Cheap or effective offers push it down (target more customers), and expensive or weak offers push it up. The model's ranking stays the same, though: only the cut-off changes. In practice the retention team would plug in their real offer cost and success rate, and the threshold would update automatically.

### 5.7 Error analysis: who do we miss?
We compare the churners the model catches (true positives) with the ones it misses (false negatives), using out-of-fold predictions on the training set.

In [ ]:
analysis = TelcoFeatures().transform(X_train).assign(
    churn=y_arr, proba=oof_best, offer=oof_best >= BEST_THRESHOLD, annual_value=value_train)
TP, FN, FP, TN = "Caught churner (TP)", "Missed churner (FN)", "False alarm (FP)", "Correctly left alone (TN)"
analysis["outcome"] = np.select(
    [analysis["churn"].eq(1) & analysis["offer"], analysis["churn"].eq(1) & ~analysis["offer"],
     analysis["churn"].eq(0) & analysis["offer"]], [TP, FN, FP], default=TN)

error_summary = analysis.groupby("outcome").agg(
    customers=("churn", "size"), avg_churn_proba=("proba", "mean"),
    total_annual_value=("annual_value", "sum")).reindex([TP, FN, FP, TN])
error_summary["cost of this error ($)"] = 0.0
error_summary.loc[FN, "cost of this error ($)"] = error_summary.loc[FN, "total_annual_value"]
error_summary.loc[FP, "cost of this error ($)"] = error_summary.loc[FP, "customers"] * OFFER_COST
display(error_summary.style.format({"avg_churn_proba": "{:.2f}", "total_annual_value": "${:,.0f}",
                                    "cost of this error ($)": "${:,.0f}"}))

churners = analysis[analysis["outcome"].isin([TP, FN])]
print("Average profile of caught vs missed churners")
display(churners.groupby("outcome")[["tenure", "MonthlyCharges", "num_addons", "price_change"]].mean().round(1))
for col in ["Contract", "PaymentMethod", "tenure_group", "TechSupport"]:
    print(f"\n{col} (share within each group)")
    display(pd.crosstab(churners["outcome"], churners[col], normalize="index").round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for name, color in [(TP, "tab:green"), (FN, "tab:red")]:
    axes[0].hist(churners.loc[churners["outcome"] == name, "proba"], bins=30, alpha=0.6, color=color, label=name)
axes[0].axvline(BEST_THRESHOLD, color="black", linestyle="--", label=f"Threshold = {BEST_THRESHOLD:.2f}")
axes[0].set(xlabel="Predicted churn probability", ylabel="Customers", title="Churners: caught vs missed")
axes[0].legend()
(pd.crosstab(churners["Contract"], churners["outcome"], normalize="columns")
   .reindex(["Month-to-month", "One year", "Two year"])
   .plot.bar(ax=axes[1], color=["tab:green", "tab:red"]))
axes[1].set(xlabel="", ylabel="Share of group", title="Contract type: caught vs missed churners")
axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

show_cols = ["tenure", "Contract", "InternetService", "PaymentMethod", "MonthlyCharges", "num_addons", "proba", "annual_value"]
print("Most valuable missed churners")
analysis[analysis["outcome"] == FN].sort_values("annual_value", ascending=False)[show_cols].head(10).round(2)

**What the errors tell us.** The churners we catch look like the textbook profile: month-to-month, new, electronic check. The ones we miss look **loyal**: longer tenure, one- or two-year contracts, automatic payment, more add-ons and often tech support. Many of them are also high-value fiber customers. Nothing in this dataset tells the model that these customers are unhappy.

A person would not catch them from these columns either. What would help is the information we listed in section 2: **contract end dates** (a two-year customer can leave when the contract expires), recent complaints or service calls, outages in their area, and competitor offers. Adding even one of these would be the most useful next step.

The false alarms are mostly month-to-month fiber customers paying by electronic check. They have the "risky" profile but stayed this month. Some of them may churn in later months, so the cost of these errors is probably overstated.

### 5.8 Explaining the final model

In [ ]:
try:
    import shap

    prep_steps, final_estimator = best_model[:-1], best_model[-1]
    X_train_t = prep_steps.transform(X_train)
    background = X_train_t.sample(100, random_state=RANDOM_STATE)
    X_explain = X_train_t.sample(500, random_state=RANDOM_STATE)

    if isinstance(final_estimator, LogisticRegression):
        explainer = shap.LinearExplainer(final_estimator, background)
        shap_values = explainer(X_explain)
    else:
        explainer = shap.Explainer(lambda a: final_estimator.predict_proba(
            pd.DataFrame(a, columns=X_train_t.columns))[:, 1], background)
        shap_values = explainer(X_explain)
    if shap_values.values.ndim == 3:
        shap_values = shap_values[..., 1]
    shap.plots.beeswarm(shap_values, max_display=15)
except ImportError:
    print("shap is not installed: pip install shap")

To make this easier to present, we add up the SHAP values within each business theme:

In [ ]:
THEMES = {
    "Contract type": ["Contract"],
    "Tenure": ["tenure", "tenure_group", "TotalCharges"],
    "Monthly price": ["MonthlyCharges", "price_change"],
    "Internet type": ["InternetService"],
    "Payment and billing": ["PaymentMethod", "auto_payment", "PaperlessBilling"],
    "Add-on services": ADDON_COLS + ["num_addons"],
    "Phone services": ["PhoneService", "MultipleLines"],
    "Demographics": ["gender", "SeniorCitizen", "Partner", "Dependents"],
}


def theme_of(feature):
    for theme, cols in THEMES.items():
        if any(feature == c or feature.startswith(c + "_") for c in cols):
            return theme
    return "Other"


if "shap_values" in globals():
    shap_df = pd.DataFrame(shap_values.values, columns=X_explain.columns)
    theme_importance = (shap_df.T.groupby(shap_df.columns.map(theme_of)).sum().T.abs().mean().sort_values())
    ax = theme_importance.plot.barh(figsize=(9, 5), color="#4C78A8")
    ax.set(xlabel="Mean |SHAP value|", ylabel="", title="What drives churn predictions? (grouped by theme)")
    plt.tight_layout()
    plt.show()

**Takeaway for the business:** contract type and tenure dominate. A new month-to-month customer is the highest-risk profile, while internet type and payment method add a smaller but clear signal. Demographics, including gender, contribute very little. SHAP shows what the model relies on, not what *causes* churn. To find out whether moving someone to a one-year contract actually keeps them, the company would need an experiment.

## 6. Final Evaluation on the Test Set
Everything above (model choice, hyperparameters, threshold) was decided using the training set only. We now score the 1,409 held-out customers **once**.

In [ ]:
y_te = y_test.to_numpy()
test_proba = best_model.predict_proba(X_test)[:, 1]


def threshold_metrics(y, proba, t):
    pred = proba >= t
    return {"Accuracy": accuracy_score(y, pred), "Recall": recall_score(y, pred),
            "Precision": precision_score(y, pred, zero_division=0), "F1": f1_score(y, pred)}


final_metrics = pd.DataFrame({
    "Train (5-fold CV)": {"PR-AUC": average_precision_score(y_arr, oof_best),
                          "ROC-AUC": roc_auc_score(y_arr, oof_best),
                          **threshold_metrics(y_arr, oof_best, BEST_THRESHOLD)},
    "Test (held out)": {"PR-AUC": average_precision_score(y_te, test_proba),
                        "ROC-AUC": roc_auc_score(y_te, test_proba),
                        **threshold_metrics(y_te, test_proba, BEST_THRESHOLD)},
})
final_metrics["Difference"] = final_metrics["Test (held out)"] - final_metrics["Train (5-fold CV)"]
print(f"Model: {best_name}  |  threshold: {BEST_THRESHOLD:.2f}")
final_metrics.round(3)

In [ ]:
test_rule = RuleBaseline().fit(X_train, y_train).predict(X_test)
tp = ((test_rule == 1) & (y_te == 1)).sum()
rule_recall, rule_precision = tp / (y_te == 1).sum(), tp / (test_rule == 1).sum()
rule_fpr = ((test_rule == 1) & (y_te == 0)).sum() / (y_te == 0).sum()

fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
ConfusionMatrixDisplay(confusion_matrix(y_te, test_proba >= BEST_THRESHOLD),
                       display_labels=["Stay", "Churn"]).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set(title=f"Confusion matrix (test, threshold {BEST_THRESHOLD:.2f})")
fpr, tpr, _ = roc_curve(y_te, test_proba)
axes[1].plot(fpr, tpr, linewidth=2, label=f"{best_name} (AUC={roc_auc_score(y_te, test_proba):.3f})")
axes[1].plot(rule_fpr, rule_recall, "ko", markersize=10, label="Rule baseline")
axes[1].plot([0, 1], [0, 1], "k--", label="Random")
axes[1].set(xlabel="False positive rate", ylabel="Recall", title="ROC curve (test)")
prec, rec, _ = precision_recall_curve(y_te, test_proba)
axes[2].plot(rec, prec, linewidth=2, label=f"{best_name} (AP={average_precision_score(y_te, test_proba):.3f})")
axes[2].plot(rule_recall, rule_precision, "ko", markersize=10, label="Rule baseline")
axes[2].axhline(y_te.mean(), color="k", linestyle="--", label="Random")
axes[2].set(xlabel="Recall", ylabel="Precision", title="Precision-recall curve (test)")
for ax in axes[1:]:
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
value_test = X_test["MonthlyCharges"].to_numpy() * CLV_MONTHS
model_row = f"Model, threshold {BEST_THRESHOLD:.2f}"
test_strategies = pd.DataFrame([
    summarize("No offers", np.zeros(len(y_te)), y_te, value_test),
    summarize("Offer to everyone", np.ones(len(y_te)), y_te, value_test),
    summarize("Rule baseline", test_rule, y_te, value_test),
    summarize("Model, threshold 0.50", test_proba >= 0.5, y_te, value_test),
    summarize(model_row, test_proba >= BEST_THRESHOLD, y_te, value_test),
    summarize("Perfect model (upper bound)", y_te == 1, y_te, value_test),
]).set_index("strategy")
display(test_strategies.style.format(FMT))

col = "savings per 1,000 customers ($)"
share_of_perfect = test_strategies.loc[model_row, col] / test_strategies.loc["Perfect model (upper bound)", col]
uplift_vs_rule = test_strategies.loc[model_row, col] / test_strategies.loc["Rule baseline", col] - 1
print(f"On the test set the model captures {share_of_perfect:.0%} of the savings a perfect model would get,")
print(f"and saves {uplift_vs_rule:.0%} more than the rule baseline.")

### How confident are we? Bootstrap interval for the savings
1,409 customers is not a huge sample, so we resample the test set 2,000 times to get a 95% interval for the savings.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n = len(y_te)
model_offer = test_proba >= BEST_THRESHOLD
boot = []
for _ in range(2000):
    idx = rng.integers(0, n, n)
    yb, vb = y_te[idx], value_test[idx]
    base = total_cost(np.zeros(n), yb, vb)
    model_s = (base - total_cost(model_offer[idx], yb, vb)) / n * 1000
    rule_s = (base - total_cost(test_rule[idx], yb, vb)) / n * 1000
    boot.append({"model": model_s, "rule": rule_s, "model minus rule": model_s - rule_s})
boot = pd.DataFrame(boot)
ci = boot.quantile([0.025, 0.5, 0.975]).T
ci.columns = ["2.5%", "median", "97.5%"]
print("Savings per 1,000 customers, 95% bootstrap interval (test set)")
display(ci.round(0))
print(f"Share of resamples where the model beats the rule: {(boot['model minus rule'] > 0).mean():.1%}")

The test results are very close to the cross-validation results, so the model generalizes and we did not overfit during selection. Even the low end of the interval is well above the rule baseline.

### Save the model for scoring new customers

In [ ]:
import joblib

os.makedirs("models", exist_ok=True)
artifact = {"model": best_model, "model_name": best_name, "threshold": BEST_THRESHOLD,
            "assumptions": {"offer_cost": OFFER_COST, "success_rate": SUCCESS_RATE, "clv_months": CLV_MONTHS},
            "input_columns": list(X_train.columns)}
joblib.dump(artifact, "models/churn_model.joblib")
loaded = joblib.load("models/churn_model.joblib")


def score_customers(frame, artifact=loaded):
    # Churn probability and offer decision for raw customer rows.
    proba = artifact["model"].predict_proba(frame[artifact["input_columns"]])[:, 1]
    return pd.DataFrame({"churn_probability": proba.round(3),
                         "send_offer": proba >= artifact["threshold"]}, index=frame.index)


new_customer = pd.DataFrame([{
    "gender": "Female", "SeniorCitizen": 0, "Partner": "No", "Dependents": "No", "tenure": 3,
    "PhoneService": "Yes", "MultipleLines": "No", "InternetService": "Fiber optic", "OnlineSecurity": "No",
    "OnlineBackup": "No", "DeviceProtection": "No", "TechSupport": "No", "StreamingTV": "Yes",
    "StreamingMovies": "Yes", "Contract": "Month-to-month", "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check", "MonthlyCharges": 95.0, "TotalCharges": 280.0}])
print("Example: a new month-to-month fiber customer")
display(score_customers(new_customer))
print("First 10 test customers")
score_customers(X_test.head(10))

## 7. Conclusions and Limitations

**What we found**
- A tuned, cost-aware churn model saves about twice as much as the simple rule or offering to everyone, and the gap holds across the bootstrap interval.
- Contract type and tenure are the strongest drivers. New month-to-month customers, especially those on fiber paying by electronic check, are the highest-risk group.
- The more complex models (random forest, boosting, the ensemble) did not clearly beat logistic regression. With a few strong, mostly linear signals, the simpler model is just as good and easier to explain.
- The best threshold depends heavily on the offer cost and success rate, so the business inputs matter as much as the model.

**Limitations and what we would do about them**
1. **One-month snapshot.** `Churn` only records who left last month, and there are no dates. Some "false alarms" may leave next month, so the real value is probably understated. *Fix:* collect monthly snapshots and train on "will this customer leave in the next 30 days", splitting by time.
2. **The model predicts churn, not response to an offer.** It finds who is likely to leave, not who would stay *because* of an offer. Some customers it targets would leave anyway, and others would have stayed without one. *Fix:* run an A/B test on offers and build an uplift model.
3. **The cost numbers are assumptions.** $50, 30% and 12 months are not measured, and monthly charges are revenue, not profit. *Fix:* the sensitivity table shows how the decision changes. The team should plug in real values and update the threshold.
4. **Blind spot for "loyal-looking" churners.** The model misses long-tenure, contract, auto-pay customers because their reason for leaving is not in the data. *Fix:* add contract end dates, service calls, complaints and outage data.
5. **Sample data.** This is an IBM sample dataset from one company, likely simplified, and it may not reflect today's market or other regions. *Fix:* retrain on the company's own recent data and check that the top drivers stay the same.
6. **Calibration.** Because of `class_weight="balanced"`, the probabilities are too high in absolute terms. This does not affect the ranking or the cost-based threshold, but the scores should not be shown to managers as literal percentages without calibrating them first.